# Week 94, Day 2: How open-source projects work

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w094d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import Counter

authors = ["ana", "ana", "ben", "ana", "chi", "ana", "ben", "ana"]
counts = Counter(authors).most_common()
print(counts, counts[0][1] / len(authors))

> **Key idea.** Learn who decides (a maintainer, a council, a foundation) and respect maintainers' time with small, tested, rule-following changes. Licences: permissive, weak copyleft, strong copyleft, by SPDX id; ask a qualified person about particular uses. Read CONTRIBUTING first; sign the CLA or DCO if asked. Triage labels issues, asks for reproductions and versions, and closes duplicates: a good first contribution. Check health: community files, recent contributors, bus factor, time to first response.

### Exercise: Sorting licences

Write `licence_families(deps)`: `deps` maps a package to its SPDX licence id. Return a dict with any of the keys `"permissive"`, `"weak copyleft"`, `"strong copyleft"` and `"unknown"` (in that order, leaving out empty ones), each mapping to the sorted packages in that family. Permissive: `MIT`, `BSD-2-Clause`, `BSD-3-Clause`, `Apache-2.0`, `ISC`, `PSF-2.0`. Weak copyleft: `LGPL-2.1-or-later`, `LGPL-3.0-or-later`, `MPL-2.0`. Strong copyleft: `GPL-2.0-only`, `GPL-2.0-or-later`, `GPL-3.0-only`, `GPL-3.0-or-later`, `AGPL-3.0-only`, `AGPL-3.0-or-later`.

In [ ]:
def licence_families(deps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_families():\n    \"a dependency list\"\n    deps = {\"requests\": \"Apache-2.0\", \"rich\": \"MIT\", \"chardet\": \"LGPL-2.1-or-later\", \"pyqt6\": \"GPL-3.0-only\",\n            \"certifi\": \"MPL-2.0\", \"mystery\": \"Custom-1.0\", \"attrs\": \"MIT\", \"typing-extensions\": \"PSF-2.0\"}\n    assert licence_families(deps) == {\"permissive\": [\"attrs\", \"requests\", \"rich\", \"typing-extensions\"],\n                                      \"weak copyleft\": [\"certifi\", \"chardet\"], \"strong copyleft\": [\"pyqt6\"], \"unknown\": [\"mystery\"]}\n    assert list(licence_families({\"x\": \"ISC\", \"y\": \"AGPL-3.0-or-later\"})) == [\"permissive\", \"strong copyleft\"]")

### Exercise: What CONTRIBUTING asks

Write `contributing_gaps(text)`: check, ignoring case, for five topics and return the missing ones in this order: `"running the tests"` (mentions `test`), `"code style"` (any of `style`, `black`, `ruff`, `flake8`, `pep 8`), `"commit messages"` (`commit`), `"licence agreement"` (any of `cla`, `contributor licence agreement`, `contributor license agreement`, `dco`, `signed-off-by`), and `"code of conduct"` (`code of conduct`). Match `cla` and `dco` as whole words.

In [ ]:
import re

def contributing_gaps(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gaps():\n    \"what a newcomer needs\"\n    good = \"\"\"# Contributing\nRun the tests with `pytest`. We format with Ruff. Write commit messages in the imperative.\nSign the CLA before your first pull request. Everyone follows our Code of Conduct.\"\"\"\n    assert contributing_gaps(good) == []\n    assert contributing_gaps(\"Please be kind. Sign-off each commit (Signed-off-by: your name), per the DCO.\") == [\n        \"running the tests\", \"code style\", \"code of conduct\"]\n    assert contributing_gaps(\"We declare nothing here.\") == [\"running the tests\", \"code style\", \"commit messages\", \"licence agreement\", \"code of conduct\"]")

### Exercise: Triage

Write `triage(issue)`: `issue` has `title` and `body`. Suggest labels in this order: `"bug"` if the title or body mentions (as whole words, ignoring case) `error`, `crash`, `crashes`, `traceback`, `fails`, `wrong` or `broken`; `"feature"` if the title starts with `add`, `support` or `allow`, or the text contains `would be nice` or `feature request`; `"question"` if the title ends with `?` or starts with `how`; and, for bugs only, `"needs reproduction"` if the body has no code block (three backticks) and `"needs version"` if the body has no version number like `3.12` (digits, a dot, digits).

In [ ]:
import re

def triage(issue):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\n\ndef test_triage():\n    \"labels and requests\"\n    assert triage({\"title\": \"Crash when the map is empty\", \"body\": \"It fails with a traceback.\"}) == [\"bug\", \"needs reproduction\", \"needs version\"]\n    assert triage({\"title\": \"Wrong time for round walks\", \"body\": f\"On 1.2.0:\\n{FENCE}\\nplan('Mill')\\n{FENCE}\"}) == [\"bug\"]\n    assert triage({\"title\": \"Add GPX export\", \"body\": \"It would be nice to have.\"}) == [\"feature\"]\n    assert triage({\"title\": \"How do I load a CSV map?\", \"body\": \"Thanks!\"}) == [\"question\"]\n    assert triage({\"title\": \"Docs typo\", \"body\": \"Erroneous word on the home page.\"}) == []")

### Exercise: Likely duplicates

Write `duplicates(issues, threshold=0.6)`: each issue is `(number, title)`. A title's words are its lower-case runs of letters and digits. For every pair of issues (in the order given, earlier first), compute the **Jaccard similarity** of their word sets (the size of the intersection over the size of the union) and return `(first, second, score)` for pairs at or above the threshold, with the score rounded to 2 decimal places.

In [ ]:
import re

def duplicates(issues, threshold=0.6):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("ISSUES = [(12, \"Crash when map is empty\"), (15, \"Add GPX export\"), (19, \"crash when the map is empty!\"),\n          (23, \"GPX export please\"), (31, \"Round walks too long\")]\n\ndef test_dupes():\n    \"close the duplicate, link the original\"\n    assert duplicates(ISSUES) == [(12, 19, 0.83)]\n    assert duplicates(ISSUES, threshold=0.5) == [(12, 19, 0.83), (15, 23, 0.5)]\n    assert duplicates([]) == []")

### Exercise: The bus factor

Write `bus_factor(authors)`: `authors` is the list of authors of a project's commits. Return `(k, people)`: the smallest number of authors who between them wrote more than half the commits, taking authors with the most commits first (ties by name), and those authors in that order. Return `(0, [])` for no commits.

In [ ]:
from collections import Counter

def bus_factor(authors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bus():\n    \"how many people could leave\"\n    assert bus_factor([\"ana\", \"ana\", \"ben\", \"ana\", \"chi\", \"ana\", \"ben\", \"ana\"]) == (1, [\"ana\"])\n    assert bus_factor([\"ana\", \"ben\", \"chi\", \"dee\", \"ana\", \"ben\"]) == (2, [\"ana\", \"ben\"])\n    assert bus_factor([\"zed\", \"amy\", \"bob\", \"cat\"]) == (3, [\"amy\", \"bob\", \"cat\"])\n    assert bus_factor([]) == (0, [])")

### Bug hunt: Nothing is ever stale

Issue 12 hasn't been touched for 40 days, but `.seconds` is only the seconds part of the difference (0 here), so this prints `[]`. Use the days, so it prints `[12]`.

In [ ]:
from datetime import date

issues = [(12, date(2026, 10, 1)), (19, date(2026, 11, 2))]
today = date(2026, 11, 10)
print([n for n, updated in issues if (today - updated).seconds > 30])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a timedelta has days\"\n    expect_output(\"[12]\")")

### Bug hunt: Similar titles that look different

Jaccard similarity divides by the size of the **union**; this divides by the two sizes added together, which counts shared words twice, and prints `0.45`. Fix it so it prints `0.83`.

In [ ]:
a = set("crash when map is empty".split())
b = set("crash when the map is empty".split())
print(round(len(a & b) / (len(a) + len(b)), 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"intersection over union\"\n    expect_output(\"0.83\")")

### Puzzle: Counting towards half

Arrange the loop of `bus_factor`: take authors with most commits first until they cover more than half. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the licence family of "Apache-2.0"
q = ___   # how many people sit on CPython's Steering Council
r = ___   # the Jaccard similarity of {"a", "b"} and {"b", "c"}, rounded to 2 places
s = ___   # True or False: code with no licence may be freely copied
t = ___   # the line a DCO adds to a commit message, up to and including its colon

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"permissive\" and q == 5 and r == 0.33, \"p, q and r\"\n    assert s is False and t == \"Signed-off-by:\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a project health report

Write `health_report(project)`. `project` has `files` (a set of community files present), `commits` (a list of `(author, "YYYY-MM-DD")`), `issues` (dicts with `number`, `opened`, `closed` (a date string or `None`), `first_response` (a date string or `None`) and `labels`), and `today`. Return `(lines, verdict)`:

1. `"files: ok"` or `"files: missing "` and the missing ones of `README`, `LICENSE`, `CONTRIBUTING`, `CODE_OF_CONDUCT` and `SECURITY`, in that order, joined by `", "`;
2. `"bus factor: K (A, B)"` (today's rules, over all commits);
3. `"active contributors (90 days): N"`: distinct authors with a commit at most 90 days before today;
4. `"open issues: N, median age D days"` (D for the open issues, `g` format; `"open issues: 0"` if none);
5. `"first response: median D days, M open without one"` (D over issues with a response, `g` format, or `-`);
6. `"good first issues: N open"` (open issues labelled `good first issue`).

`verdict` is `"healthy"` if no files are missing, the bus factor is at least 2, at least 2 contributors are active, and the median first response is at most 7 days; otherwise `"at risk: "` and the failing ones of `files`, `bus factor`, `activity`, `response time`, joined by `", "`.

In [ ]:
from collections import Counter
from datetime import date

def health_report(project):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def issue(number, opened, closed=None, first_response=None, labels=()):\n    return {\"number\": number, \"opened\": opened, \"closed\": closed, \"first_response\": first_response, \"labels\": list(labels)}\n\nPROJECT = {\n    \"files\": {\"README\", \"LICENSE\", \"CODE_OF_CONDUCT\"},\n    \"commits\": [(\"ana\", \"2026-11-01\"), (\"ana\", \"2026-10-20\"), (\"ben\", \"2026-10-02\"), (\"ana\", \"2026-06-01\"),\n                (\"chi\", \"2026-03-15\"), (\"ana\", \"2026-02-01\")],\n    \"issues\": [issue(12, \"2026-09-01\", first_response=\"2026-09-03\"), issue(15, \"2026-10-01\", first_response=\"2026-10-21\", labels=[\"good first issue\"]),\n               issue(19, \"2026-11-01\"), issue(23, \"2026-06-01\", closed=\"2026-06-10\", first_response=\"2026-06-02\"),\n               issue(31, \"2026-11-08\", labels=[\"good first issue\", \"docs\"])],\n    \"today\": \"2026-11-10\",\n}\n\ndef test_report():\n    \"a small project, examined\"\n    lines, verdict = health_report(PROJECT)\n    assert lines == [\n        \"files: missing CONTRIBUTING, SECURITY\",\n        \"bus factor: 1 (ana)\",\n        \"active contributors (90 days): 2\",\n        \"open issues: 4, median age 24.5 days\",\n        \"first response: median 2 days, 2 open without one\",\n        \"good first issues: 2 open\",\n    ]\n    assert verdict == \"at risk: files, bus factor\"\n\ndef test_healthy():\n    \"everything in order\"\n    project = {\"files\": {\"README\", \"LICENSE\", \"CONTRIBUTING\", \"CODE_OF_CONDUCT\", \"SECURITY\"},\n               \"commits\": [(\"ana\", \"2026-11-01\"), (\"ben\", \"2026-11-02\")], \"issues\": [], \"today\": \"2026-11-10\"}\n    lines, verdict = health_report(project)\n    assert lines == [\"files: ok\", \"bus factor: 2 (ana, ben)\", \"active contributors (90 days): 2\", \"open issues: 0\",\n                     \"first response: median - days, 0 open without one\", \"good first issues: 0 open\"]\n    assert verdict == \"healthy\"\n    project[\"commits\"] = [(\"ana\", \"2026-11-01\")]\n    assert health_report(project)[1] == \"at risk: bus factor, activity\"")

## Warm-up

One person wrote more than half of these commits. If Ana stopped, the project would be in trouble: that's what the **bus factor** measures (the smallest number of people who between them wrote more than half the code, so their leaving would leave nobody who knows how it works), and it's one of the first things worth checking about a project you're thinking of contributing to.

## Who decides

Every project has some **governance**, written down or not:

- A **single maintainer** (common for small libraries): one person decides, often in their spare time.
- A **core team** or **steering council**: CPython is run by a five-person Steering Council elected by the core developers (PEP 13), with core developers who can merge changes and a much larger group of contributors.
- A **foundation** behind it: the Python Software Foundation holds CPython's intellectual property and runs PyPI and PyCon US.

Whatever the shape, maintainers' time is the scarcest thing in open source. Small, well-explained, well-tested contributions that follow the project's rules use little of it, which is why they get merged.

## Licences

Code without a licence isn't open source: by default nobody else may copy it. Licences, named by short **SPDX** identifiers, fall into families:

- **Permissive** (`MIT`, `BSD-3-Clause`, `Apache-2.0`, CPython's `PSF-2.0`): use it for anything, keep the notice.
- **Weak copyleft** (`LGPL-3.0-or-later`, `MPL-2.0`): changes to the licensed files themselves must stay open; using it from your code is usually fine.
- **Strong copyleft** (`GPL-3.0-or-later`, `AGPL-3.0-or-later`): a program that includes it, when distributed (or for AGPL, offered over a network), must be released under the same licence.

A tool can sort your dependencies into these families, which is useful for spotting surprises. What a licence means for a particular use is a legal question: ask someone qualified, not a script. Many projects also ask contributors to sign a **CLA** (Contributor Licence Agreement; CPython uses the PSF's) or to certify a **DCO** (Developer Certificate of Origin) with a `Signed-off-by:` line.

## Contributing and triage

A project's `CONTRIBUTING` file is its instructions for you: how to set up, how to run the tests, the code style, how to write commit messages, the licence agreement, and the code of conduct. Read it before writing anything, and do what it says.

Issues arrive faster than maintainers can read them, so projects **triage**: label each issue (bug, feature request, question), ask for what's missing (steps to reproduce, the version), and close **duplicates** with a link to the original. Triage is also one of the most useful first contributions you can make: reproducing a reported bug, or finding the duplicate, saves a maintainer real time.

## Is it healthy?

Before investing weeks, check the project is alive and welcoming: the community files exist (README, LICENSE, CONTRIBUTING, CODE_OF_CONDUCT, SECURITY), more than one person has commits recently, the **bus factor** is above one, and new issues get a **first response** within days, not months. GitHub's "community standards" page and the CHAOSS project's metrics (free) measure these in more detail.

## Practice

## Quiz

## Daily challenge

## Your turn on a real project

1. **Governance and licence**: for the project you chose yesterday, find who can merge changes, how decisions are made, and its licence.
2. **Read its CONTRIBUTING** and list exactly what it asks of you; if there's a CLA or DCO, find out how to sign it.
3. **Triage three issues**: try to reproduce one bug, and search for a duplicate of another. If you can add something useful (a reproduction, a version, a link to the duplicate), comment politely.
4. **Run a health check** with your own version of `health_report`, from what the repository's pages show.

## Before you go

Tomorrow turns to CPython itself: its developer guide, how its repository is laid out, and how a change gets from an issue to a merged pull request.